# Keyword Detection with PyTorch
This notebook implements a Convolutional Neural Network (CNN) to detect specific keywords from audio files. It demonstrates the process of loading and preprocessing `.wav` files using `torchaudio`, defining a custom PyTorch dataset, applying data augmentation, training a classification model, and finally exporting the optimized model to ONNX format for deployment.

---
### 1. Imports & Environment
Grouped imports for standard libraries, third-party packages, and local modules. We also enable inline plotting for matplotlib.

In [ ]:
%matplotlib inline

# Standard library imports
import os
import sys
import json
import time
import copy
import random
import wave
from dataclasses import dataclass
from pathlib import Path

# Third-party imports
import numpy as np
import soundfile as sf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# PyTorch imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchaudio

# ONNX imports
import onnxruntime as ort
import warnings

---
### 2. Check GPU Status
Ensure that PyTorch has access to the GPU for faster training.

In [ ]:
def check_gpu_status():
    """
    Checks if a CUDA-compatible GPU is available and prints its properties.
    Defaults to CPU if no GPU is found.
    """
    cuda_available = torch.cuda.is_available()
    print(f"CUDA Available: {cuda_available}")

    if cuda_available:
        num_gpus = torch.cuda.device_count()
        print(f"Number of GPUs detected: {num_gpus}\n")
        for i in range(num_gpus):
            print(f"--- GPU {i} ---")
            print(f"Name: {torch.cuda.get_device_name(i)}")
            total_memory = torch.cuda.get_device_properties(i).total_memory / (1024**3)
            print(f"Total Memory: {total_memory:.2f} GB")
    else:
        print("PyTorch cannot detect a compatible GPU. It will default to CPU.")

check_gpu_status()

---
### 3. Configuration
Define hyper-parameters and load configurations. We use a `dataclass` to organize our settings cleanly.

In [ ]:
# Add parent directory to path to load local Utils if necessary
sys.path.append(os.path.abspath('..'))
try:
    from Utils.config_loader import get_keywords, get_config_value
except ImportError:
    # Dummy fallbacks in case Utils is not available
    def get_keywords(): 
        print("Getting Classes from Backup {ImportError}")
        return ["yes", "no", "up", "down", "other"]
    def get_config_value(key, default): return default

def load_keywords():
    """Attempts to find and load config.json to get keywords."""
    # Using pathlib for cleaner path resolution
    current_dir = Path.cwd()
    for p in [current_dir / "config.json", current_dir.parent / "config.json", current_dir.parent.parent / "config.json"]:
        if p.exists():
            with open(p, "r") as f:
                print("Getting Classes from Config.json")
                return json.load(f).get("keywords", ["yes", "no", "up", "down", "other"])
    return get_keywords()

@dataclass
class Config:
    """Hyperparameters and configuration settings."""
    CLASSES: list = None
    TARGET_SAMPLE_RATE: int = get_config_value('target_sample_rate', 16000)
    NUM_SAMPLES: int = get_config_value('num_samples', 16000)
    BATCH_SIZE: int = 32
    EPOCHS: int = 200
    LEARNING_RATE: float = 0.001
    EARLY_STOPPING_PATIENCE: int = 4       # To Disable:    EARLY_STOPPING_PATIENCE: int = EPOCHS
    DEVICE: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    DATA_DIR: Path = Path("../dataset/")
    MODEL_DIR: Path = Path("Models")
    MODEL_PATH_PTH: Path = MODEL_DIR.joinpath("PyTorch.pth")
    MODEL_PATH_ONNX: Path = MODEL_DIR.joinpath("PyTorch.onnx")
    print(f"Models: \n  - {MODEL_PATH_ONNX} \n  - { MODEL_PATH_PTH }")
    
    def __post_init__(self):
        if self.CLASSES is None:
            self.CLASSES = load_keywords()
            os.makedirs(self.MODEL_DIR, exist_ok=True)

cfg = Config()
print(f"Classes: {cfg.CLASSES}")
print(f"Device: {cfg.DEVICE}")

---
### 4. Load Dataset
Scan the dataset directory and split the files into training and testing sets.

In [ ]:
def get_files_and_labels(data_dir, classes):
    """
    Scans the data directory for audio files matching the target classes.
    """
    file_paths = []
    labels = []
    class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}

    for cls_name in classes:
        cls_dir = data_dir / cls_name
        if not cls_dir.exists() or not any(cls_dir.iterdir()):
            raise FileNotFoundError(
                f"\n{'='*60}\n"
                f"DATASET MISSING: The directory for class '{cls_name}' was not found or is empty.\n"
                f"Please ensure you have created the folder '{cls_dir}' and added .wav files to it.\n"
                f"Alternatively, update your config.json keywords to match existing folders.\n"
                f"{'='*60}\n"
            )
            
        for file in cls_dir.iterdir():
            if file.suffix == ".wav":
                file_paths.append(str(file))
                labels.append(class_to_idx[cls_name])

    return file_paths, labels

file_paths, labels = get_files_and_labels(cfg.DATA_DIR, cfg.CLASSES)
if not file_paths:
    print("Error: No files found! Check your DATA_DIR.")
else:
    train_paths, test_paths, train_labels, test_labels = train_test_split(
        file_paths, labels, test_size=0.2, stratify=labels, random_state=42
    )
    print(f"Dataset split: {len(train_paths)} train, {len(test_paths)} test")

---
### 5. Model Definition
Define the custom dataset for audio loading and augmentation, and construct the CNN architecture.

In [ ]:
class KeywordDataset(Dataset):
    """
    Custom Dataset for loading keyword audio files.
    Applies data augmentation (random shifts) during training.
    """
    def __init__(self, paths, labels, config, is_training=False):
        self.paths = paths
        self.labels = labels
        self.config = config
        self.is_training = is_training

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        clean_path = os.path.normpath(self.paths[idx])
        waveform_np, sr = sf.read(clean_path, dtype="float32")
        waveform = torch.from_numpy(waveform_np)

        # Ensure correct channel dimension [1, time]
        if waveform.ndim == 1:
            waveform = waveform.unsqueeze(0)
        else:
            waveform = waveform.t()

        # Convert stereo to mono by averaging channels
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        if self.is_training:
            # Augmentation: Random audio shift with zero-padding (instead of roll)
            shift = random.randint(-1600, 1600)
            if shift > 0:
                waveform = F.pad(waveform, (shift, 0))[:, :-shift]
            elif shift < 0:
                waveform = F.pad(waveform, (0, -shift))[:, -shift:]

        # Truncate or pad to exactly NUM_SAMPLES
        if waveform.shape[1] > self.config.NUM_SAMPLES:
            if self.is_training:
                # Random crop for training
                start = random.randint(0, waveform.shape[1] - self.config.NUM_SAMPLES)
                waveform = waveform[:, start:start + self.config.NUM_SAMPLES]
            else:
                # Center crop for evaluation
                start = (waveform.shape[1] - self.config.NUM_SAMPLES) // 2
                waveform = waveform[:, start:start + self.config.NUM_SAMPLES]
        elif waveform.shape[1] < self.config.NUM_SAMPLES:
            waveform = F.pad(waveform, (0, self.config.NUM_SAMPLES - waveform.shape[1]))

        return waveform, torch.tensor(self.labels[idx], dtype=torch.long)

class KeywordCNN(nn.Module):
    """
    Convolutional Neural Network for Audio Keyword Classification.
    Expects MFCC spectrograms as input.
    """
    def __init__(self, num_classes):
        super(KeywordCNN, self).__init__()
        # 4 Convolutional layers to extract time-frequency features
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)
        # Adaptive pooling allows for variable input lengths (though we pad/crop to fixed size)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))

        self.fc1 = nn.Linear(128 * 4 * 4, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.pool(F.relu(self.conv4(x)))
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(F.relu(self.fc1(x)))
        logits = self.fc2(x)
        return logits

---
### 6. Evaluation Helper
A DRY (Don't Repeat Yourself) helper function for running inference on a given dataloader.

In [ ]:
def evaluate_model(model, dataloader, mfcc_transform, criterion=None, device=cfg.DEVICE):
    """
    Helper function to evaluate the model on a given dataloader.
    Returns average loss, accuracy, all predictions, and all targets.
    """
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_targets = [], []
    
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            inputs = mfcc_transform(inputs)
            outputs = model(inputs)
            
            if criterion is not None:
                loss = criterion(outputs, targets)
                total_loss += loss.item()
                
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    acc = 100. * correct / total
    avg_loss = total_loss / len(dataloader) if criterion else 0.0
    return avg_loss, acc, all_preds, all_targets

---
### 7. Training & Main Loop
Instantiate the transforms, setup the optimizer, and train the model with early stopping.

In [ ]:
def train_model(config, patience=15):
    """
    Trains the CNN model, handles early stopping, and saves the best weights.
    """
    train_loader = DataLoader(KeywordDataset(train_paths, train_labels, config, is_training=True), 
                              batch_size=config.BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    test_loader = DataLoader(KeywordDataset(test_paths, test_labels, config, is_training=False), 
                             batch_size=config.BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    model = KeywordCNN(num_classes=len(config.CLASSES)).to(config.DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=cfg.EARLY_STOPPING_PATIENCE)

    mfcc_transform = torchaudio.transforms.MFCC(
        sample_rate=config.TARGET_SAMPLE_RATE, n_mfcc=40, melkwargs={"n_mels": 64}
    ).to(config.DEVICE)
    
    freq_masking = torchaudio.transforms.FrequencyMasking(freq_mask_param=10).to(config.DEVICE)
    time_masking = torchaudio.transforms.TimeMasking(time_mask_param=20).to(config.DEVICE)

    best_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())
    early_stop_counter = 0

    history = {
        'train_loss': [], 'test_loss': [],
        'train_acc': [],  'test_acc': []
    }

    print(f"Training on {config.DEVICE}...")

    for epoch in range(config.EPOCHS):
        start_time = time.time()
        model.train()
        total_loss, correct, total = 0, 0, 0

        for batch_idx, (inputs, targets) in enumerate(train_loader):
            inputs, targets = inputs.to(config.DEVICE), targets.to(config.DEVICE)
            with torch.no_grad():
                inputs = mfcc_transform(inputs)
                inputs = freq_masking(inputs)
                inputs = time_masking(inputs)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

        train_acc = 100. * correct / total
        train_loss = total_loss / len(train_loader)

        # Use helper for evaluation
        test_avg_loss, test_acc, _, _ = evaluate_model(model, test_loader, mfcc_transform, criterion, config.DEVICE)
        
        # Save history
        history['train_loss'].append(train_loss)
        history['test_loss'].append(test_avg_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)

        dur = time.time() - start_time
        print(f"Epoch {epoch+1}/{config.EPOCHS} [{dur:.1f}s] | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

        if test_acc > best_acc:
            best_acc = test_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(best_model_wts, cfg.MODEL_PATH_PTH)
            early_stop_counter = 0
        else:
            early_stop_counter += 1
            if early_stop_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
        scheduler.step(test_acc)

    model.load_state_dict(best_model_wts)
    return model, history, mfcc_transform

if __name__ == '__main__':
    # We also return mfcc_transform for later use in evaluation
    best_model, history, mfcc_transform = train_model(cfg, patience=cfg.EARLY_STOPPING_PATIENCE)

---
### 8. Export to ONNX
Programmatically determine the dummy input shape to safely export the model to ONNX.

In [ ]:
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*version conversion.*")

dynamic_axes = {
    'mfcc_input': {0: 'batch_size', 3: 'time'},
    'logits': {0: 'batch_size'}
}

best_model.eval()

# Derive dummy input dynamically from one batch
dummy_loader = DataLoader(KeywordDataset(train_paths[:1], train_labels[:1], cfg, is_training=False), batch_size=1)
sample_input, _ = next(iter(dummy_loader))
with torch.no_grad():
    dummy_input = mfcc_transform(sample_input.to(cfg.DEVICE))

print(f"Exporting model with dummy input shape: {dummy_input.shape}...")

try:
    torch.onnx.export(
        best_model,
        dummy_input,
        str(cfg.MODEL_PATH_ONNX),
        export_params=True,
        opset_version=18,     # Set to 18 for broad modern ONNX compatibility
        do_constant_folding=True,
        input_names=['mfcc_input'],
        output_names=['logits'],
        dynamic_axes=dynamic_axes,
        verbose=False
    )
    print(f"✅ SUCCESS: {cfg.MODEL_PATH_ONNX}")
except Exception as e:
    print(f"⚠️ FAILED: {e}")

---
### 9. Visualize Training
Plot the accuracy and loss curves over the training epochs.

In [ ]:
plt.figure(figsize=(12, 5))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['test_acc'], label='Test Acc')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['test_loss'], label='Test Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

---
### 10. Confusion Matrix
Evaluate the model on the full test set and visualize class confusions.

In [ ]:
def show_confusion_matrix(model, transform):
    """Evaluates the model and displays a confusion matrix."""

    test_loader = DataLoader(KeywordDataset(test_paths, test_labels, cfg), batch_size=cfg.BATCH_SIZE, shuffle=False)
    full_loader = DataLoader(KeywordDataset(file_paths, labels, cfg), batch_size=cfg.BATCH_SIZE, shuffle=False)

    _, _, all_preds, all_targets = evaluate_model(model, test_loader, transform, device=cfg.DEVICE)
    cm = confusion_matrix(all_targets, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=cfg.CLASSES, yticklabels=cfg.CLASSES)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix Test')

    _, _, all_preds, all_targets = evaluate_model(model, full_loader, transform, device=cfg.DEVICE)
    cm = confusion_matrix(all_targets, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=cfg.CLASSES, yticklabels=cfg.CLASSES)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix Full')

    plt.show()

show_confusion_matrix(best_model, mfcc_transform)

---
### 11. Failed Files Analysis
List up to a specified number of files that the model predicted incorrectly.

In [ ]:
from IPython.display import Audio, display
from pathlib import Path

def list_failed_files(model, transform, limit=-1):
    """Prints a list of test files that were classified incorrectly with audio players."""
    test_loader = DataLoader(KeywordDataset(test_paths, test_labels, cfg), batch_size=cfg.BATCH_SIZE, shuffle=False)
    _, _, all_preds, all_targets = evaluate_model(model, test_loader, transform, device=cfg.DEVICE)
    
    failed_count = 0
    print(f"{'True Label':<12} | {'Predicted':<12} | {'File Path'}")
    print("-" * 60)
    
    for i, (pred, target) in enumerate(zip(all_preds, all_targets)):
        if limit != -1 and failed_count >= limit:
            break
            
        if pred != target:
            true_label = cfg.CLASSES[target]
            pred_label = cfg.CLASSES[pred]
            file_path = test_paths[i]
            
            print(f"{true_label:<12} | {pred_label:<12} | {Path(file_path).name}")
            display(Audio(filename=file_path))
            failed_count += 1

list_failed_files(best_model, mfcc_transform, limit=-1)

---
### 12. Class Distribution & Training Volume
This analysis shows the balance of our classes and the total number of training examples the model will encounter.

In [ ]:
import matplotlib.pyplot as plt
import os

def analyze_distribution():
    counts = {cls: len([f for f in os.listdir(os.path.join(cfg.DATA_DIR, cls)) if f.endswith('.wav')]) 
              for cls in cfg.CLASSES if os.path.isdir(os.path.join(cfg.DATA_DIR, cls))}
    
    total = sum(counts.values())
    effective = total * cfg.EPOCHS
    
    print(f"Total Raw Samples: {total:,}")
    print(f"Effective Samples (with Augmentation over {cfg.EPOCHS} epochs): {effective:,}")
    
    plt.figure(figsize=(10, 4))
    plt.bar(counts.keys(), counts.values(), color='teal')
    plt.title("Samples per Keyword")
    plt.show()

analyze_distribution()

---
### 13. Duration Statistics
We'll check the length of all audio files to ensure consistency. 

In [ ]:
import wave
import numpy as np

def analyze_time():
    durations = []
    for cls in cfg.CLASSES:
        cls_path = os.path.join(cfg.DATA_DIR, cls)
        if not os.path.isdir(cls_path): continue
        for f in os.listdir(cls_path):
            if f.endswith('.wav'):
                with wave.open(os.path.join(cls_path, f), 'rb') as w:
                    durations.append(w.getnframes() / w.getframerate())
    
    total_sec = sum(durations)
    print(f"Total Dataset Duration: {total_sec/60:.2f} minutes")
    print(f"Average Duration: {np.mean(durations):.4f}s")
    print(f"Shortest/Longest: {min(durations):.4f}s / {max(durations):.4f}s")

    plt.figure(figsize=(10, 4))
    plt.hist(durations, bins=30, color='orange', edgecolor='black')
    plt.title("Histogram of File Durations")
    plt.xlabel("Seconds")
    plt.show()

analyze_time()

---
### 14. Technical File Integrity
Verification of sample rates and audio properties to prevent training crashes.

In [ ]:
def analyze_specs():
    sr_counts = {}
    mismatches = []
    
    for cls in cfg.CLASSES:
        cls_path = os.path.join(cfg.DATA_DIR, cls)
        if not os.path.isdir(cls_path): continue
        for f in os.listdir(cls_path):
            if f.endswith('.wav'):
                f_path = os.path.join(cls_path, f)
                with wave.open(f_path, 'rb') as w:
                    sr = w.getframerate()
                    sr_counts[sr] = sr_counts.get(sr, 0) + 1
                    if sr != cfg.TARGET_SAMPLE_RATE:
                        mismatches.append(f_path)
    
    print("--- Sample Rate Distribution ---")
    for sr, count in sr_counts.items():
        print(f"{sr} Hz: {count} files")
    
    if mismatches:
        print(f"\n⚠️ FOUND {len(mismatches)} FILES WITH WRONG SAMPLE RATE!")
        print(f"Example: {mismatches[0]}")
    else:
        print("\n✅ All files match TARGET_SAMPLE_RATE.")

analyze_specs()